# Toàn bộ query → SubData → KDL trên Google Colab

Notebook này chạy hai pha liên tiếp:

1. Đọc **toàn bộ query**, route corpus/document/page và tạo một thư mục SubData riêng cho từng query.
2. Chạy pipeline KDL tuần tự trên từng SubData: `ingestion → cleaning → enrichment → chunking/embedding → integration → artifacts`.

Mặc định KDL chỉ xử lý các PDF một trang trong `pages/`; full document vẫn được lưu trong `documents/`. Mỗi query có `kdl_status.json`, log riêng và có thể resume sau khi Colab bị ngắt.

## 1. Kiểm tra GPU và cài đặt


In [ ]:
import os
import torch

assert torch.cuda.is_available(), "Không tìm thấy CUDA. Hãy chọn GPU runtime."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print(f"CPU: {os.cpu_count() or 1}")

In [ ]:
from pathlib import Path
import subprocess
import sys

DISCOVERY_REPO = Path("/content/data-discovery")
AXIOM_REPO = Path("/content/AXIOM_DE-RD")

if not (DISCOVERY_REPO / ".git").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ManhTanTran/data-discovery.git",
        str(DISCOVERY_REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(DISCOVERY_REPO), "pull", "--ff-only"], check=True)

if not (AXIOM_REPO / ".git").exists():
    subprocess.run([
        "git", "clone", "--branch", "baseline", "--single-branch",
        "https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git",
        str(AXIOM_REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(AXIOM_REPO), "fetch", "origin", "baseline"], check=True)
    subprocess.run(["git", "-C", str(AXIOM_REPO), "checkout", "baseline"], check=True)
    subprocess.run(["git", "-C", str(AXIOM_REPO), "pull", "--ff-only", "origin", "baseline"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{DISCOVERY_REPO}[ml]"], check=True)

if str(DISCOVERY_REPO) not in sys.path:
    sys.path.insert(0, str(DISCOVERY_REPO))
from sentence_transformers import SentenceTransformer
print("Đã cài đặt và kiểm tra Sentence Transformers thành công.")
print("KDL/vLLM sẽ được cài sau khi đã tạo xong toàn bộ SubData.")

## 2. Kết nối Drive và cấu hình thí nghiệm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MY_DRIVE = Path("/content/drive/MyDrive")
DATA_ROOT = MY_DRIVE / "AXIOM_DE-RD" / "data"

# Chỉ cần điền hai biến override nếu cấu trúc Drive của bạn khác.
DATA_DIR_OVERRIDE = ""
QUERY_SOURCE_OVERRIDE = ""
EXISTING_BATCH_DIR = ""  # Điền batch cũ để bỏ qua pha tạo SubData và resume KDL.

MAX_QUERIES = None          # None = tạo SubData cho toàn bộ query.
MAX_KDL_QUERIES = None      # None = chạy KDL cho toàn bộ SubData.
START_KDL_FROM = 1          # Bắt đầu từ query thứ mấy, tính từ 1.
KDL_INPUT_MODE = "pages"  # "pages" (khuyến nghị) hoặc "documents".
RESUME_KDL = True           # Bỏ qua query đã có kdl_status.json = success.

if DATA_DIR_OVERRIDE:
    data_dir = Path(DATA_DIR_OVERRIDE)
else:
    candidates = [
        MY_DRIVE / "vidore_v3_industrial" / "pdfs",
        MY_DRIVE / "iSE_DE" / "vidore_v3" / "vidore_v3_industrial" / "pdfs",
    ]
    data_dir = next((path for path in candidates if path.exists()), None)
    if data_dir is None:
        matches = list(MY_DRIVE.glob("**/vidore_v3_industrial/pdfs"))
        data_dir = matches[0] if matches else None

assert data_dir is not None and data_dir.is_dir(), (
    "Không tìm thấy thư mục PDF. Hãy đặt DATA_DIR_OVERRIDE."
)
query_source = Path(QUERY_SOURCE_OVERRIDE) if QUERY_SOURCE_OVERRIDE else data_dir.parent / "queries"
assert query_source.exists(), f"Không tìm thấy nguồn query: {query_source}"
assert DATA_ROOT.parent.exists(), (
    f"Không tìm thấy {DATA_ROOT.parent}. Hãy tạo shortcut AXIOM_DE-RD vào My Drive."
)
assert KDL_INPUT_MODE in {"pages", "documents"}

print(f"Thư mục PDF gốc: {data_dir}")
print(f"Nguồn query: {query_source}")
print(f"Thư mục dữ liệu AXIOM: {DATA_ROOT}")
print(f"KDL sẽ xử lý: {KDL_INPUT_MODE}")

## 3. Tạo Light Index và SubData cho toàn bộ query


In [ ]:
import time
import pandas as pd
from IPython.display import display
from src.data_discovery import (
    DiscoveryConfig, LightIndex, LightPreparer, QueryRouter,
    SentenceTransformerEmbedder, load_queries, run_query_batch,
)

config = DiscoveryConfig(
    top_k_corpora=1,
    top_k_documents=8,
    top_k_pages=12,
    late_interaction_top_k=3,
    selection_threshold=0.15,
    alpha=0.25, beta=0.60, gamma=0.15,
    exploration_rate=0.05,
    max_preview_chars=1200,
    max_preview_segments_per_document=64,
    create_pdf_thumbnails=False,
    ann_backend="faiss",
)

if EXISTING_BATCH_DIR:
    batch_dir = Path(EXISTING_BATCH_DIR)
    assert batch_dir.is_dir(), f"Không tìm thấy batch: {batch_dir}"
    summary_path = batch_dir / "batch_summary.csv"
    assert summary_path.is_file(), f"Thiếu batch_summary.csv trong {batch_dir}"
    summary_df = pd.read_csv(summary_path)
    print(f"Dùng lại batch hiện có: {batch_dir}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name_light = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    offline_started = time.perf_counter()
    preparation_started = time.perf_counter()
    manifest = LightPreparer(config).prepare({"vidore_industrial": data_dir})
    light_preparation_ms = (time.perf_counter() - preparation_started) * 1000

    index_started = time.perf_counter()
    light_embedder = SentenceTransformerEmbedder(model_name_light, device=device, batch_size=64)
    light_index = LightIndex(manifest, light_embedder, ann_backend=config.ann_backend)
    light_index_ms = (time.perf_counter() - index_started) * 1000
    offline_timing_ms = {
        "light_preparation_ms": light_preparation_ms,
        "light_index_ms": light_index_ms,
        "total_offline_ms": (time.perf_counter() - offline_started) * 1000,
    }

    queries = load_queries(query_source)
    queries_to_run = queries if MAX_QUERIES is None else queries[:MAX_QUERIES]
    print(f"Tổng query đọc được: {len(queries)}")
    print(f"Số query sẽ tạo SubData: {len(queries_to_run)}")

    result = run_query_batch(
        QueryRouter(light_index, config), manifest, queries_to_run,
        DATA_ROOT / "output", config,
        corpus_name="vidore_v3_industrial",
        copy_documents=True,
        extract_pages=True,
        offline_timing_ms=offline_timing_ms,
    )
    batch_dir = Path(result.batch_dir)
    summary_path = Path(result.summary_path)
    summary_df = pd.DataFrame(result.rows)

print(f"Batch SubData: {batch_dir}")
print(f"Số query trong batch: {len(summary_df)}")
print(f"SubData thành công: {(summary_df['status'] == 'success').sum()}")
display(summary_df.head(10))

## 4. Giải phóng model nhẹ trước khi serve KDL


In [ ]:
import gc

for variable_name in ("light_index", "light_embedder"):
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng model embedding nhẹ khỏi GPU.")

### Cài KDL/vLLM sau pha Data Discovery

Thứ tự này là bắt buộc trên Colab: vLLM thay đổi bộ PyTorch/quantization dependency và có thể làm `sentence-transformers` không import được nếu được cài trước pha Light Index.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "vllm==0.19.0"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{AXIOM_REPO}[pdf-inspector]"],
    check=True,
)
print("Đã cài đặt AXIOM, KDL và vLLM sau khi hoàn tất SubData.")

## 5. Serve KDL bằng vLLM trên chính Colab


In [ ]:
from google.colab import userdata

try:
    VLLM_API_KEY = userdata.get("VLLM_API_KEY")
except Exception as exc:
    VLLM_API_KEY = None
    print(f"Không đọc được Colab Secret ({type(exc).__name__}); dùng khóa nội bộ.")
if not VLLM_API_KEY:
    VLLM_API_KEY = "kdl-local-colab"
    print("Đang dùng khóa vLLM nội bộ vì endpoint chỉ chạy trên localhost.")
os.environ["VLLM_API_KEY"] = VLLM_API_KEY

gpu_name = gpu.name
vram_gib = gpu.total_memory / 1024**3
cpu_count = os.cpu_count() or 1
if ("H100" in gpu_name) or vram_gib >= 90:
    dtype, max_seqs, max_tokens, gpu_util = "bfloat16", 256, 65536, 0.90
    render_processes = min(32, cpu_count)
elif "A100" in gpu_name and vram_gib >= 70:
    dtype, max_seqs, max_tokens, gpu_util = "bfloat16", 128, 32768, 0.90
    render_processes = min(24, cpu_count)
elif "A100" in gpu_name:
    dtype, max_seqs, max_tokens, gpu_util = "bfloat16", 64, 16384, 0.95
    render_processes = min(16, cpu_count)
elif "L4" in gpu_name:
    dtype, max_seqs, max_tokens, gpu_util = "bfloat16", 64, 16384, 0.90
    render_processes = min(12, cpu_count)
else:
    dtype, max_seqs, max_tokens, gpu_util = "float16", 8, 4096, 0.90
    render_processes = min(4, cpu_count)

vllm_log = Path("/content/vllm-kdl.log")
vllm_pid_file = Path("/content/vllm.pid")
vllm_running = False
if vllm_pid_file.exists():
    try:
        os.kill(int(vllm_pid_file.read_text().strip()), 0)
        vllm_running = True
    except (OSError, ValueError):
        pass

if not vllm_running:
    log_handle = vllm_log.open("w", encoding="utf-8")
    command = [
        "vllm", "serve", "KDLAI/KDL-Frontier-Parser-nano",
        "--host", "0.0.0.0", "--port", "8000",
        "--api-key", VLLM_API_KEY,
        "--served-model-name", "kdl-frontier-parser-nano",
        "--dtype", dtype, "--max-model-len", "8192",
        "--max-num-seqs", str(max_seqs),
        "--max-num-batched-tokens", str(max_tokens),
        "--gpu-memory-utilization", str(gpu_util),
        "--limit-mm-per-prompt", '{"image":1}',
        "--trust-remote-code", "--enable-chunked-prefill",
        "--enable-prefix-caching", "--generation-config", "vllm",
    ]
    vllm_process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT)
    log_handle.close()
    vllm_pid_file.write_text(str(vllm_process.pid))
    print(f"Đã khởi động vLLM, PID={vllm_process.pid}")
else:
    print(f"vLLM đang chạy, PID={vllm_pid_file.read_text().strip()}")

In [ ]:
import requests

VLLM_API_BASE = "http://127.0.0.1:8000/v1"
headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
deadline = time.time() + 30 * 60
while time.time() < deadline:
    try:
        response = requests.get(f"{VLLM_API_BASE}/models", headers=headers, timeout=5)
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(10)
else:
    print(vllm_log.read_text(errors="replace")[-20000:])
    raise RuntimeError("vLLM chưa sẵn sàng sau 30 phút.")

model_name = response.json()["data"][0]["id"]
os.environ["VLLM_API_BASE"] = VLLM_API_BASE
os.environ["VLLM_MODEL_NAME"] = model_name
print(f"KDL vLLM đã sẵn sàng: {model_name}")

## 6. Chạy KDL tuần tự cho từng SubData


In [ ]:
import json
from datetime import datetime

config_candidates = [
    AXIOM_REPO / "configs/pipeline.vidore-v3-physics-kdl-pdf-inspector.yaml",
    AXIOM_REPO / "configs/pipeline.kdl-pdf-inspector.yaml",
]
source_config = next((path for path in config_candidates if path.is_file()), None)
assert source_config is not None, "Không tìm thấy cấu hình KDL trong AXIOM_DE-RD."
base_config = json.loads(source_config.read_text(encoding="utf-8"))

embedder_name = str(base_config.get("chunking_embedding", {}).get("embedder", ""))
if "openrouter" in embedder_name.lower():
    openrouter_key = userdata.get("OPENROUTER_API_KEY")
    assert openrouter_key, "Cấu hình embedding cần OPENROUTER_API_KEY trong Colab Secrets."
    os.environ["OPENROUTER_API_KEY"] = openrouter_key

query_dirs = sorted(
    path for path in batch_dir.iterdir()
    if path.is_dir() and (path / "subdata_manifest.json").is_file()
)
query_dirs = query_dirs[max(START_KDL_FROM - 1, 0):]
if MAX_KDL_QUERIES is not None:
    query_dirs = query_dirs[:MAX_KDL_QUERIES]

print(f"Cấu hình KDL gốc: {source_config}")
print(f"Số SubData sẽ duyệt qua: {len(query_dirs)}")
print(f"Chế độ input: {KDL_INPUT_MODE}")

In [ ]:
import copy
import shutil

def đọc_tóm_tắt_embedding(metadata_path):
    if not metadata_path.is_file():
        return {}
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    return metadata.get("summary", {}).get("embedding", {})

def tìm_run_mới_nhất(stage_dir):
    runs = sorted(
        (path for path in stage_dir.iterdir() if path.is_dir()),
        key=lambda path: path.stat().st_mtime,
    ) if stage_dir.is_dir() else []
    return runs[-1] if runs else None

summary_df = pd.read_csv(summary_path)
for column in ("kdl_status", "kdl_input_count", "kdl_run_id", "kdl_ms",
               "embedding_generated_count", "total_to_kdl_ms", "kdl_error"):
    if column not in summary_df.columns:
        summary_df[column] = ""

for ordinal, query_dir in enumerate(query_dirs, start=1):
    status_path = query_dir / "kdl_status.json"
    if RESUME_KDL and status_path.is_file():
        old_status = json.loads(status_path.read_text(encoding="utf-8"))
        if old_status.get("status") == "success":
            print(f"[{ordinal}/{len(query_dirs)}] Bỏ qua {query_dir.name}: đã thành công.")
            continue

    input_dir = query_dir / KDL_INPUT_MODE
    input_files = sorted(input_dir.glob("*.pdf")) if input_dir.is_dir() else []
    started = time.perf_counter()
    started_at = datetime.now().isoformat(timespec="seconds")
    status = {
        "status": "running", "started_at": started_at,
        "input_mode": KDL_INPUT_MODE, "input_count": len(input_files),
        "query_dir": str(query_dir),
    }
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")

    if not input_files:
        status.update({"status": "failed", "error": "Không có PDF đầu vào cho KDL."})
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[{ordinal}/{len(query_dirs)}] Lỗi {query_dir.name}: không có PDF.")
        continue

    config_kdl = copy.deepcopy(base_config)
    config_kdl["local_input"]["path"] = str(input_dir)
    config_kdl["local_input"]["recursive"] = False
    config_kdl["local_input"]["max_files"] = None
    kdl_root = query_dir / "kdl"
    for stage in ("ingested", "cleaned", "enriched", "embedded", "output"):
        config_kdl[f"{stage}_dir"] = str(kdl_root / stage)

    kdl_config = config_kdl["parsing"]["kdl"]
    kdl_config["endpoint_url"] = VLLM_API_BASE
    kdl_config["model"] = model_name
    kdl_config["render_processes"] = render_processes
    kdl_config["output_dir"] = str(kdl_root / "work")
    kdl_config["max_workers"] = min(32, max_seqs)
    kdl_config["bbox_max_workers"] = min(32, max_seqs)

    embed_params = config_kdl.get("chunking_embedding", {}).get("embedder_params", {})
    if "cache_dir" in embed_params:
        embed_params["cache_dir"] = str(DATA_ROOT / "work/embedding_cache/full-query-kdl")

    runtime_config = query_dir / "kdl_runtime_config.json"
    runtime_config.write_text(json.dumps(config_kdl, ensure_ascii=False, indent=2), encoding="utf-8")
    log_path = query_dir / "kdl_pipeline.log"

    try:
        with log_path.open("w", encoding="utf-8") as log_handle:
            completed = subprocess.run(
                [sys.executable, "-u", "scripts/run_pipeline.py", "--config", str(runtime_config)],
                cwd=str(AXIOM_REPO), env=os.environ.copy(),
                stdout=log_handle, stderr=subprocess.STDOUT,
            )

        embedded_run = tìm_run_mới_nhất(kdl_root / "embedded")
        output_run = tìm_run_mới_nhất(kdl_root / "output")
        run_id = embedded_run.name if embedded_run else ""
        embedding_summary = đọc_tóm_tắt_embedding(
            embedded_run / "metadata.json" if embedded_run else Path("/khong-ton-tai")
        )
        generated_count = int(embedding_summary.get("generated_count", 0) or 0)
        output_count = len(list((output_run / "documents").glob("*.json"))) if output_run else 0
        elapsed_ms = (time.perf_counter() - started) * 1000
        success = completed.returncode == 0 and output_count == len(input_files) and generated_count > 0
        status.update({
            "status": "success" if success else "failed",
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "return_code": completed.returncode, "run_id": run_id,
            "output_document_count": output_count,
            "embedding_generated_count": generated_count,
            "kdl_ms": elapsed_ms, "log_path": str(log_path),
        })
        if not success:
            status["error"] = (
                f"return_code={completed.returncode}, output={output_count}/{len(input_files)}, "
                f"embedding_generated={generated_count}"
            )
    except Exception as exc:
        elapsed_ms = (time.perf_counter() - started) * 1000
        generated_count, run_id = 0, ""
        status.update({
            "status": "failed", "finished_at": datetime.now().isoformat(timespec="seconds"),
            "kdl_ms": elapsed_ms, "error": f"{type(exc).__name__}: {exc}",
        })

    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
    row_mask = summary_df["query_dir"].astype(str) == str(query_dir)
    query_to_subdata = pd.to_numeric(summary_df.loc[row_mask, "query_to_subdata_ms"], errors="coerce").fillna(0)
    summary_df.loc[row_mask, "kdl_status"] = status["status"]
    summary_df.loc[row_mask, "kdl_input_count"] = len(input_files)
    summary_df.loc[row_mask, "kdl_run_id"] = status.get("run_id", "")
    summary_df.loc[row_mask, "kdl_ms"] = status.get("kdl_ms", 0)
    summary_df.loc[row_mask, "embedding_generated_count"] = status.get("embedding_generated_count", 0)
    summary_df.loc[row_mask, "total_to_kdl_ms"] = query_to_subdata + status.get("kdl_ms", 0)
    summary_df.loc[row_mask, "kdl_error"] = status.get("error", "")
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(
        f"[{ordinal}/{len(query_dirs)}] {query_dir.name}: {status['status']} | "
        f"input={len(input_files)} | embedding={status.get('embedding_generated_count', 0)} | "
        f"{status.get('kdl_ms', 0) / 1000:.1f} giây"
    )

print("Đã kết thúc vòng lặp KDL.")

## 7. Tổng hợp kết quả


In [ ]:
summary_df = pd.read_csv(summary_path)
kdl_success = summary_df[summary_df.get("kdl_status", "") == "success"]
kdl_failed = summary_df[summary_df.get("kdl_status", "") == "failed"]

print(f"Batch: {batch_dir}")
print(f"Tổng query: {len(summary_df)}")
print(f"KDL thành công: {len(kdl_success)}")
print(f"KDL thất bại: {len(kdl_failed)}")
print(f"Chưa chạy KDL: {len(summary_df) - len(kdl_success) - len(kdl_failed)}")
if not kdl_success.empty:
    print(f"KDL trung bình: {pd.to_numeric(kdl_success['kdl_ms']).mean() / 1000:.1f} giây/query")
    print(f"Query → SubData → KDL trung bình: {pd.to_numeric(kdl_success['total_to_kdl_ms']).mean() / 1000:.1f} giây/query")
display(summary_df[[
    "query_id", "status", "selected_documents", "selected_pages",
    "kdl_status", "kdl_input_count", "embedding_generated_count",
    "query_to_subdata_ms", "kdl_ms", "total_to_kdl_ms", "kdl_error",
]].head(50))
print(f"File tổng hợp: {summary_path}")

## Cấu trúc output cho mỗi query

```text
batch_<timestamp>/<query_id>/
├── documents/                 # Full document được chọn
├── pages/                     # Page PDF được chọn
├── subdata_manifest.json
├── selected_pages.csv
├── kdl_runtime_config.json
├── kdl_pipeline.log
├── kdl_status.json
└── kdl/
    ├── ingested/<run_id>/
    ├── cleaned/<run_id>/
    ├── enriched/<run_id>/
    ├── embedded/<run_id>/
    └── output/<run_id>/
```

Nếu runtime bị ngắt, đặt `EXISTING_BATCH_DIR` thành đường dẫn batch này, giữ `RESUME_KDL = True` và chạy lại notebook.